In [8]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load the trained 3D CNN model
model = load_model('models/fight_detection_model.h5')  # Replace with your model's path on Kaggle/Device

# Function to test the model on a video and overlay "FIGHT" text when violence is detected
def test_on_video(input_video_path, output_video_path):
    # Open the input video
    video = cv2.VideoCapture(input_video_path)

    # Get video properties
    frame_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(video.get(cv2.CAP_PROP_FPS))

    # Video writer to save the output video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    # Sliding window to collect frames
    sliding_window = []
    max_frames = 64  # Matching the input shape of the model

    while video.isOpened():
        ret, frame = video.read()

        if not ret:
            break  # Stop if video ends

        # Preprocess the frame (resize and normalize)
        resized_frame = cv2.resize(frame, (224, 224))  # Resize frame to match the input size
        resized_frame = resized_frame / 255.0  # Normalize pixel values to [0,1]
        sliding_window.append(resized_frame)

        # If we have enough frames for a prediction (e.g., 64 frames for 3D CNN)
        if len(sliding_window) == max_frames:
            # Prepare the batch of frames for the model
            input_frames = np.array(sliding_window)
            input_frames = np.expand_dims(input_frames, axis=0)  # Add batch dimension
            input_frames = input_frames.astype('float32')

            # Make prediction
            prediction = model.predict(input_frames)
            predicted_class = np.argmax(prediction, axis=1)[0]  # 0: No Fight, 1: Fight

            # If "Violence" (fight) is detected, overlay red "FIGHT" text at the lower center
            if predicted_class == 1:  # Violence class
                # Get text size and calculate lower center position
                text = "FIGHT"
                font_scale = 3  # Reduced font size
                font_thickness = 3  # Reduced boldness
                font = cv2.FONT_HERSHEY_SIMPLEX
                text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]

                # Set the position for the text
                text_x = (frame_width - text_size[0]) // 2  # Center horizontally
                text_y = frame_height - 30  # Closer to bottom (30 pixels from the bottom)

                # Define the rectangle parameters with reduced size
                rect_width = text_size[0] + 10  # Smaller width of the rectangle with padding
                rect_height = text_size[1] + 10  # Smaller height of the rectangle with padding
                rect_x = text_x - 5  # Rectangle's top-left x position
                rect_y = text_y - text_size[1] - 5  # Rectangle's top-left y position

                # Draw a transparent rectangle with a red outline
                overlay = frame.copy()  # Make a copy of the frame to draw on
                alpha = 0.1  # Transparency factor
                cv2.rectangle(overlay, (rect_x, rect_y), (rect_x + rect_width, rect_y + rect_height), (0, 0, 255), -1)  # Red filled rectangle
                cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)  # Add transparency

                # Draw the red outline for the rectangle
                cv2.rectangle(frame, (rect_x, rect_y), (rect_x + rect_width, rect_y + rect_height), (0, 0, 255), 1)  # Thinner red outline

                # Overlay the red text with anti-aliasing
                cv2.putText(frame, text, (text_x, text_y), font, font_scale, (0, 0, 255), font_thickness, cv2.LINE_AA)

            # Write the frame to the output video
            out.write(frame)

            # Remove the oldest frame from the sliding window (for real-time processing)
            sliding_window.pop(0)

    # Release the video objects
    video.release()
    out.release()

    print(f"Test completed. The video is saved at {output_video_path}")

# Example usage
input_video_path = 'data/input/non-violence/t_n001_converted.avi'  # Replace with your input video path on Kaggle
output_video_path = 'data/output/output2.avi'  # Path to save the output video on Kaggle

test_on_video(input_video_path, output_video_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 410ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 767ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 